# 2 - Merge the LoRA adapter into the base model, then push to the HF Hub  *(optional)*

Only needed if you want a **standalone** model. The Gradio demo (notebook 3) does **not** need this.

**The merged model is ~14.5 GB - it is NOT written to Google Drive** (a 15 GB Drive can't hold it).
It is written to Colab's local disk (`/content`, ~100 GB, wiped when the session ends) and uploaded
straight to a Hugging Face model repo.

**Runtime > Restart session first** - the fp16 load needs a clean T4 (~14.5 / 16 GB).

In [ ]:
!pip -q install -U "transformers>=4.44,<5" "peft>=0.12" "accelerate>=0.34" "safetensors>=0.4" "huggingface_hub>=0.24"

# Colab preinstalls an old torchao (0.10.x). The peft version above calls its LoRA dispatcher
# for every target module, and that dispatcher raises ImportError on a too-old torchao instead
# of just skipping it -- even though this plain fp16/4bit merge never touches torchao at all.
# Uninstalling it makes peft's is_torchao_available() return False cleanly (package not found)
# and skip that dispatcher, instead of crashing on the version check.
!pip -q uninstall -y torchao

### Mount Drive and confirm the adapter folder
Mounted explicitly here (before it's referenced) so `ls` below actually works.
Run this cell, check the printed folder names, then set `ADAPTER` in the next cell to match exactly
what you see -- don't guess the name.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!ls "/content/drive/MyDrive/" | grep -i psycho

In [ ]:
import torch, os, shutil
from huggingface_hub import login, create_repo, upload_folder
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))    # token needs WRITE permission
except Exception:
    login()

BASE        = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER     = "/content/drive/MyDrive/psycho_mistral_v03_transformed_adapter"  # <-- confirmed via ls above
MERGED_REPO = "antfr99/psycho-mistral-v03-merged"                   # <-- HF model repo to create / push to
LOCAL_OUT   = "/content/psycho_mistral_v03_merged"                  # Colab local disk (ephemeral) - NOT Drive
MERGED_PRIVATE = True

assert os.path.isfile(os.path.join(ADAPTER, "adapter_config.json")), (
    f"No adapter_config.json found in {ADAPTER} -- re-check the folder name with the `ls` cell above "
    "before running the merge."
)

for p in ["/content", "/content/drive/MyDrive"]:
    try:
        t, u, f = shutil.disk_usage(p)
        print(f"{p}: {f/1e9:5.1f} GB free / {t/1e9:.0f} GB")
    except Exception as e:
        print(p, "->", e)
print("\nmerged model ~14.5 GB -> /content (local) -> uploaded to the Hub. Nothing big touches Drive.")

### Primary - fp16 merge (best quality)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map={"": 0},
)
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()

os.makedirs(LOCAL_OUT, exist_ok=True)
model.save_pretrained(LOCAL_OUT, safe_serialization=True, max_shard_size="4GB")
AutoTokenizer.from_pretrained(BASE).save_pretrained(LOCAL_OUT)
print("saved locally ->", LOCAL_OUT)

create_repo(MERGED_REPO, repo_type="model", private=MERGED_PRIVATE, exist_ok=True)
upload_folder(folder_path=LOCAL_OUT, repo_id=MERGED_REPO, repo_type="model",
              commit_message="merge QLoRA adapter into Mistral-7B-Instruct-v0.3")
print("pushed -> https://huggingface.co/" + MERGED_REPO)

# shutil.rmtree(LOCAL_OUT)   # uncomment to reclaim local disk before doing anything else

### Fallback - 4-bit merge
Run this **instead of the cell above** only if it raised CUDA OOM. Always fits (~6 GB); negligible
quality cost since the adapter was trained on the 4-bit base anyway.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                        device_map={"": 0}, torch_dtype=torch.float16)
m = PeftModel.from_pretrained(m, ADAPTER)
m = m.merge_and_unload()          # peft dequantizes the base to fp16 during the merge
m = m.to(torch.float16)

os.makedirs(LOCAL_OUT, exist_ok=True)
m.save_pretrained(LOCAL_OUT, safe_serialization=True, max_shard_size="4GB")
AutoTokenizer.from_pretrained(BASE).save_pretrained(LOCAL_OUT)
create_repo(MERGED_REPO, repo_type="model", private=MERGED_PRIVATE, exist_ok=True)
upload_folder(folder_path=LOCAL_OUT, repo_id=MERGED_REPO, repo_type="model",
              commit_message="merge QLoRA adapter (4-bit path) into Mistral-7B-Instruct-v0.3")
print("pushed -> https://huggingface.co/" + MERGED_REPO)